#Проверка 1

## Проверка сервера

In [ ]:
import json, time, os
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from google.colab import userdata

In [ ]:
BASE  = userdata.get("PROSPER_API_URL").rstrip("/")
TOKEN = userdata.get("PROSPER_API_TOKEN")

In [ ]:
try:
    with urlopen(BASE + "/health", timeout=20) as r:
        data = json.load(r)
        print("HTTP", r.status)
        print(json.dumps(data, indent=2, ensure_ascii=False))
        print("\n✅ calculation_verified =", data.get("calculation_verified"))
except HTTPError as e:
    print("HTTP", e.code, e.read().decode()[:300])
except Exception as e:
    print("❌", type(e).__name__, e)

HTTP 200
{
  "api": "ready",
  "ipm_files": "present",
  "calculation_verified": true,
  "apps": [
    "PROSPER"
  ],
  "max_in_flight": 3
}

✅ calculation_verified = True


##Общий клиент

In [ ]:
def api(method, path, *, session=None, body=None):
    headers = {"Authorization": f"Bearer {TOKEN}"}
    if session:
        headers["X-Session-ID"] = session
    data = None if body is None else json.dumps(body).encode()
    if data is not None:
        headers["Content-Type"] = "application/json"
    req = Request(BASE + path, data=data, headers=headers, method=method)
    try:
        with urlopen(req, timeout=20) as r:
            return r.status, json.load(r)
    except HTTPError as e:
        raw = e.read().decode(errors="replace")
        try:
            return e.code, json.loads(raw)
        except ValueError:
            return e.code, {"error": raw[:300]}

##Проверка модели

In [ ]:
status, health = api("GET", "/health")
print("health:", status, health)

status, opened = api("POST", "/v1/session")
print("session:", status, opened)
SESSION = opened.get("session_id") if status == 200 else None
print("SESSION_ID =", SESSION)

health: 200 {'api': 'ready', 'ipm_files': 'present', 'calculation_verified': True, 'apps': ['PROSPER'], 'max_in_flight': 3}
session: 200 {'session_id': 'GjtEPWOnorMsIVgnpdSzqiu8FvDxkUc4gucxpD654aE', 'expires_after_seconds': 1800}
SESSION_ID = GjtEPWOnorMsIVgnpdSzqiu8FvDxkUc4gucxpD654aE


In [ ]:
status, model = api("GET", "/v1/model", session=SESSION)
print("GET model:", status, model)

if status == 404:
    status, model = api("POST", "/v1/model", session=SESSION, body={"app": "PROSPER"})
    print("POST model:", status, model)

GET model: 200 {'app': 'PROSPER', 'created': 1795773474.3106098}


##Чтение параметра

In [ ]:
status, result = api("POST", "/v1/jobs", session=SESSION,
                     body={"op": "get", "tag": "PROSPER.PVT.Input.Solgor"})
print("queued:", status, result)
JOB_ID = result["job_id"]

for _ in range(120):
    st, cur = api("GET", f"/v1/jobs/{JOB_ID}", session=SESSION)
    if cur["status"] in ("done", "failed"):
        break
    time.sleep(1)

print("final:", st, cur)

queued: 202 {'job_id': '59b78a1c87747362beab799c20fae04c', 'status': 'queued'}
final: 200 {'job_id': '59b78a1c87747362beab799c20fae04c', 'app': 'PROSPER', 'status': 'done', 'result': {'value': '500.000000000'}, 'created': 1795790771.20416, 'updated': 1795790772.8378053}


##Закрытие сессии

In [ ]:
status, closed = api("DELETE", "/v1/session", session=SESSION)
print("closed:", status, closed)

closed: 200 {'closed': True}


# ЛЛМ

##Установка SDK и настройка

In [61]:
import os
from openai import OpenAI

# ВАЖНО: В Colab os.environ может не видеть секреты напрямую.
# Если код коллеги работает у неё локально или в другом окружении,
# а в Colab os.environ пуст, мы продублируем секреты в os.environ.
from google.colab import userdata

# Аккуратно переносим секреты Colab в переменные окружения (как у коллеги)
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
os.environ["DEEPSEEK_MODEL"] = userdata.get("DEEPSEEK_MODEL")

# Инициализация клиента как у коллеги
client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://routerai.ru/api/v1"
)

print("Клиент создан. Модель:", os.environ["DEEPSEEK_MODEL"])

Клиент создан. Модель: deepseek/deepseek-v4-pro-0813


##Оборачиваем вызовы PROSPER в инструменты

In [59]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_prosper_parameter",
            "description": "Читает значение параметра из модели PROSPER по его тегу. Например, 'PROSPER.PVT.Input.Solgor'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tag": {
                        "type": "string",
                        "description": "Полный тег параметра, например 'PROSPER.PVT.Input.Solgor'"
                    }
                },
                "required": ["tag"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_prosper_parameter",
            "description": "Устанавливает значение параметра в модели PROSPER. Изменяет общую модель.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tag": {
                        "type": "string",
                        "description": "Полный тег параметра"
                    },
                    "value": {
                        "type": "string", # Передаем строкой, PROSPER сам распарсит
                        "description": "Новое значение параметра (число или строка)"
                    }
                },
                "required": ["tag", "value"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_prosper_calculation",
            "description": "Запускает расчет в PROSPER. Доступные команды: pvt_calculate (PVT), inflow_calculate (IPR), system_calculate (IPR+VLP).",
            "parameters": {
                "type": "object",
                "properties": {
                    "command_id": {
                        "type": "string",
                        "enum": ["pvt_calculate", "inflow_calculate", "system_calculate"],
                        "description": "Идентификатор команды для запуска"
                    }
                },
                "required": ["command_id"]
            }
        }
    }
]

## Runner для инструментов

In [62]:
import json
import time

# Вспомогательная функция для запуска job и ожидания результата (PROSPER)
def execute_job(operation, session_id):
    status, queued = api("POST", "/v1/jobs", session=session_id, body=operation)
    if status != 202:
        return f"Ошибка запуска: HTTP {status}, {queued}"

    job_id = queued['job_id']
    for _ in range(60):
        st, cur = api("GET", f"/v1/jobs/{job_id}", session=session_id)
        if st != 200:
            return f"Ошибка чтения статуса: {st}, {cur}"
        if cur['status'] == 'done':
            return cur['result']
        if cur['status'] == 'failed':
            return f"Расчет не выполнен (job_id: {job_id})"
        time.sleep(1)
    return "Таймаут ожидания job"

# Функция-диспетчер
def call_tool(tool_call, session_id):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    print(f"   🤖 Агент вызывает: {name} с аргументами {args}")

    if name == "get_prosper_parameter":
        result = execute_job({"op": "get", "tag": args["tag"]}, session_id)
    elif name == "set_prosper_parameter":
        result = execute_job({"op": "set", "tag": args["tag"], "value": args["value"]}, session_id)
    elif name == "run_prosper_calculation":
        result = execute_job({"op": "run", "command_id": args["command_id"]}, session_id)
    else:
        result = f"Неизвестный инструмент: {name}"

    return str(result)

## Agent Loop

In [63]:
def run_agent(user_prompt, session_id, history=None, max_steps=5):
    messages = [
        {"role": "system", "content": "Ты — инженерный ассистент. Используй инструменты PROSPER для расчетов. Не придумывай значения, всегда вызывай инструменты."},
        {"role": "user", "content": user_prompt}
    ]
    if history:
        messages = [messages[0]] + history + [messages[1]]

    for step in range(max_steps):
        print(f"\n--- Шаг {step+1} ---")

        # Вызов ровно как в рабочем коде коллеги
        response = client.chat.completions.create(
            model=os.environ["DEEPSEEK_MODEL"],
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.1
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                tool_result = call_tool(tool_call, session_id)
                print(f"   ✅ Результат: {tool_result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_call.function.name,
                    "content": tool_result
                })
        else:
            print("\n🤖 Ответ агента:")
            return response_message.content

    return "Агент превысил лимит шагов."

## Test

In [64]:
# 1. Открываем сессию
status, opened = api("POST", "/v1/session")
if status == 200:
    session_id = opened["session_id"]
    print(f"Сессия открыта: {session_id}")

    try:
        # 2. Спрашиваем агента
        answer = run_agent(
            "Прочитай параметр PROSPER.PVT.Input.Solgor и скажи мне его значение.",
            session_id
        )
        print(f"\n🎯 Финальный ответ: {answer}")

    finally:
        # 3. Закрываем сессию
        print("\nЗакрываем сессию...")
        api("DELETE", "/v1/session", session=session_id)
        print("Сессия закрыта.")
else:
    print(f"Не удалось открыть сессию: {status}, {opened}")

Сессия открыта: SJl3jYZVEL96gfEy5sGtDjY0BbPjMrxGwAnXNT93wl0

--- Шаг 1 ---
   🤖 Агент вызывает: get_prosper_parameter с аргументами {'tag': 'PROSPER.PVT.Input.Solgor'}
   ✅ Результат: {'value': '500.000000000'}

--- Шаг 2 ---

🤖 Ответ агента:

🎯 Финальный ответ: Значение параметра **PROSPER.PVT.Input.Solgor** = **500.0** (500.000000000).

Закрываем сессию...
Сессия закрыта.


#Класс ProsperClient

In [69]:
import json
import time
from urllib.error import HTTPError
from urllib.request import Request, urlopen

class ProsperClient:
    def __init__(self, base_url: str, token: str):
        self.base = base_url.rstrip("/")
        self.token = token
        self.session_id = None

    def _api(self, method, path, *, body=None, session=None):
        headers = {"Authorization": f"Bearer {self.token}"}
        if session:
            headers["X-Session-ID"] = session
        data = None if body is None else json.dumps(body).encode("utf-8")
        if data is not None:
            headers["Content-Type"] = "application/json"

        req = Request(self.base + path, data=data, headers=headers, method=method)
        try:
            with urlopen(req, timeout=20) as response:
                return response.status, json.load(response)
        except HTTPError as error:
            raw = error.read().decode("utf-8", errors="replace")
            try:
                return error.code, json.loads(raw)
            except ValueError:
                return error.code, {"error": raw[:200]}

    def health(self):
        return self._api("GET", "/health")

    def open_session(self):
        status, opened = self._api("POST", "/v1/session")
        if status == 200:
            self.session_id = opened["session_id"]
            print(f"Сессия открыта: {self.session_id}")
            return True
        print(f"Не удалось открыть сессию: {status}, {opened}")
        return False

    def close_session(self):
        if self.session_id:
            self._api("DELETE", "/v1/session", session=self.session_id)
            print("Сессия закрыта.")
            self.session_id = None

    def execute_job(self, operation):
        if not self.session_id:
            return "Ошибка: Сессия не открыта"

        status, queued = self._api("POST", "/v1/jobs", session=self.session_id, body=operation)
        if status != 202:
            return f"Ошибка запуска: HTTP {status}, {queued}"

        job_id = queued['job_id']
        for _ in range(60):
            st, cur = self._api("GET", f"/v1/jobs/{job_id}", session=self.session_id)
            if st != 200:
                return f"Ошибка чтения статуса: {st}, {cur}"
            if cur['status'] == 'done':
                return cur['result']
            if cur['status'] == 'failed':
                return f"Расчет не выполнен (job_id: {job_id})"
            time.sleep(1)
        return "Таймаут ожидания job"

##Проверяем что лаботает (без LLM)

In [70]:
from google.colab import userdata

# Берем настройки из секретов (замени имена, если они у тебя другие)
PROSPER_URL = userdata.get("PROSPER_API_URL")
PROSPER_TOKEN = userdata.get("PROSPER_API_TOKEN")

# Создаем клиент
prosper = ProsperClient(base_url=PROSPER_URL, token=PROSPER_TOKEN)

# 1. Проверка здоровья
status, health = prosper.health()
print("Health:", status, health)

# 2. Открываем сессию
if prosper.open_session():
    try:
        # 3. Пробуем прочитать параметр через новый метод
        print("\nЧитаем Solgor...")
        result = prosper.execute_job({"op": "get", "tag": "PROSPER.PVT.Input.Solgor"})
        print("Результат:", result)
    finally:
        # 4. Обязательно закрываем сессию
        prosper.close_session()

Health: 200 {'api': 'ready', 'ipm_files': 'present', 'calculation_verified': True, 'apps': ['PROSPER'], 'max_in_flight': 3}
Сессия открыта: sBZWgW_N6i1yzbCFoMpFHxzm1Fzhccjmb8RCIr2UuBo

Читаем Solgor...
Результат: {'value': '500.000000000'}
Сессия закрыта.


##Инструменты (Tools) для агента

In [71]:
def get_tools():
    return [
        {
            "type": "function",
            "function": {
                "name": "get_prosper_parameter",
                "description": "Читает значение параметра из модели PROSPER по его тегу. Например, 'PROSPER.PVT.Input.Solgor'.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "tag": {"type": "string", "description": "Полный тег параметра"}
                    },
                    "required": ["tag"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "set_prosper_parameter",
                "description": "Устанавливает значение параметра в модели PROSPER.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "tag": {"type": "string", "description": "Полный тег параметра"},
                        "value": {"type": "string", "description": "Новое значение"}
                    },
                    "required": ["tag", "value"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "run_prosper_calculation",
                "description": "Запускает расчет в PROSPER (pvt_calculate, inflow_calculate, system_calculate).",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "command_id": {
                            "type": "string",
                            "enum": ["pvt_calculate", "inflow_calculate", "system_calculate"]
                        }
                    },
                    "required": ["command_id"]
                }
            }
        }
    ]

def call_tool(tool_call, prosper_client):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    print(f"   🤖 Агент вызывает: {name} с аргументами {args}")

    if name == "get_prosper_parameter":
        result = prosper_client.execute_job({"op": "get", "tag": args["tag"]})
    elif name == "set_prosper_parameter":
        result = prosper_client.execute_job({"op": "set", "tag": args["tag"], "value": args["value"]})
    elif name == "run_prosper_calculation":
        result = prosper_client.execute_job({"op": "run", "command_id": args["command_id"]})
    else:
        result = f"Неизвестный инструмент: {name}"

    return str(result)

##Сам класс

In [72]:
from openai import OpenAI
import os

class ProsperAgent:
    def __init__(self, api_key: str, base_url: str, model_name: str, prosper_client):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model_name
        self.prosper = prosper_client
        self.tools = get_tools()

    def run(self, user_prompt: str, max_steps: int = 5) -> str:
        messages = [
            {
                "role": "system",
                "content": "Ты — инженерный ассистент. Используй инструменты PROSPER. Не придумывай значения, всегда вызывай инструменты."
            },
            {"role": "user", "content": user_prompt}
        ]

        for step in range(max_steps):
            print(f"\n--- Шаг {step + 1} ---")
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=self.tools,
                tool_choice="auto",
                temperature=0.1
            )

            response_message = response.choices[0].message
            messages.append(response_message)

            if response_message.tool_calls:
                for tool_call in response_message.tool_calls:
                    tool_result = call_tool(tool_call, self.prosper)
                    print(f"   ✅ Результат: {tool_result}")
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": tool_result
                    })
            else:
                return response_message.content

        return "Агент превысил лимит шагов."

##Проверка

In [73]:
from google.colab import userdata
import os

# Настройки (берем из секретов Colab)
PROSPER_URL = userdata.get("PROSPER_API_URL")
PROSPER_TOKEN = userdata.get("PROSPER_API_TOKEN")
DEEPSEEK_KEY = userdata.get("DEEPSEEK_API_KEY")
DEEPSEEK_MODEL = userdata.get("DEEPSEEK_MODEL")

# 1. Инициализируем клиент PROSPER
prosper = ProsperClient(base_url=PROSPER_URL, token=PROSPER_TOKEN)

# 2. Проверяем здоровье и открываем сессию
if prosper.health()[0] == 200 and prosper.open_session():
    try:
        # 3. Создаем агента
        agent = ProsperAgent(
            api_key=DEEPSEEK_KEY,
            base_url="https://routerai.ru/api/v1",
            model_name=DEEPSEEK_MODEL,
            prosper_client=prosper
        )

        # 4. Спрашиваем
        answer = agent.run("Прочитай параметр PROSPER.PVT.Input.Solgor и скажи мне его значение.")
        print(f"\n🎯 Финальный ответ: {answer}")

    finally:
        # 5. Обязательно закрываем сессию
        prosper.close_session()
else:
    print("Не удалось запустить PROSPER.")

Сессия открыта: 22J033Wd6ZTFw9smX1uadHH4JpGL6geCVN3-YOgFbfA

--- Шаг 1 ---
   🤖 Агент вызывает: get_prosper_parameter с аргументами {'tag': 'PROSPER.PVT.Input.Solgor'}
   ✅ Результат: {'value': '500.000000000'}

--- Шаг 2 ---

🎯 Финальный ответ: Значение параметра `PROSPER.PVT.Input.Solgor` — **500.0** (в выдаче модели: `500.000000000`).
Сессия закрыта.
